# TEAM-EVAL — DEV_L21_150 Finalization

Canonicalize the original human-authored L21 draft using exact raw-frame decode and the BTC `frame_idx` coordinate contract. No query generation or semantic rewriting occurs.

Required Kaggle inputs:
1. Raw/support corpus: `/kaggle/input/datasets/nadkli/dataset-aic` (nested/symlink mount supported).
2. L21 draft dataset: `/kaggle/input/datasets/irthn1311/aic2026-dev-l21-150-draft-v1`. It may contain the original ZIP or Kaggle-expanded JSON/JSONL files.
3. Optional DEV_CROSS_60 dataset: `/kaggle/input/datasets/irthn1311/aic2026-dev-cross-60-v1`. It may likewise contain the ZIP or expanded files; it is required only for the combined development bundle.

Internet is required only to clone branch `TRIAGEEG`. No model download, GPU, or network inference is used.

Outputs:
- `/kaggle/working/aic2026_dev_l21_150_v1.zip` when L21 is scoring-ready.
- `/kaggle/working/aic2026_team_eval_dev_v1.zip` when L21 is ready and DEV_CROSS_60 input is valid.
- `/kaggle/working/aic2026_dev_l21_150_review_v1.zip` only when unresolved anchors need review.

In [ ]:
import json, os, shutil, subprocess, sys
from pathlib import Path
from zipfile import ZipFile

REPO_URL = os.environ.get('AIC_REPO_URL', 'https://github.com/Irthn1311/AIC2026_TeamPTK_SGU.git')
REPO_REF = os.environ.get('AIC_REPO_REF', 'TRIAGEEG')
REPO_DIR = Path(os.environ.get('AIC_REPO_DIR', '/kaggle/working/AIC2026_TeamPTK_SGU'))
REFRESH_REPO = os.environ.get('AIC_REFRESH_REPO', '0') == '1'
DATA_INPUT = Path(os.environ.get('AIC_DATA_ROOT', '/kaggle/input/datasets/nadkli/dataset-aic'))
DRAFT_INPUT = Path(os.environ.get('AIC_L21_DRAFT_ROOT', '/kaggle/input/datasets/irthn1311/aic2026-dev-l21-150-draft-v1'))
CROSS_INPUT = Path(os.environ.get('AIC_DEV_CROSS_ROOT', '/kaggle/input/datasets/irthn1311/aic2026-dev-cross-60-v1'))
OUTPUT_ROOT = Path('/kaggle/working/aic2026_team_eval_l21_finalize')
L21_ZIP = Path('/kaggle/working/aic2026_dev_l21_150_v1.zip')
DEV_ZIP = Path('/kaggle/working/aic2026_team_eval_dev_v1.zip')
print({'repo_url': REPO_URL, 'repo_ref': REPO_REF, 'dataset': str(DATA_INPUT), 'draft_input': str(DRAFT_INPUT), 'dev_cross_input': str(CROSS_INPUT), 'l21_zip': str(L21_ZIP), 'dev_zip': str(DEV_ZIP), 'internet_only_for_clone': True, 'gpu_required': False, 'model_download_required': False})

In [ ]:
if REFRESH_REPO and REPO_DIR.exists():
    if REPO_DIR.parent != Path('/kaggle/working') or REPO_DIR.name != 'AIC2026_TeamPTK_SGU': raise RuntimeError(f'Refusing repository cleanup outside expected Kaggle path: {REPO_DIR}')
    shutil.rmtree(REPO_DIR)
if not REPO_DIR.exists(): subprocess.run(['git','clone','--depth','1','--branch',REPO_REF,REPO_URL,str(REPO_DIR)], check=True)
if not (REPO_DIR / 'src/aic2026_eval/l21_finalize.py').is_file(): raise RuntimeError('Cloned TRIAGEEG ref does not contain L21 finalization code')
REPO_ROOT = REPO_DIR.resolve()
sys.path.insert(0, str(REPO_ROOT / 'src'))
COMMIT = subprocess.run(['git','rev-parse','HEAD'], cwd=REPO_ROOT, capture_output=True, text=True, check=True).stdout.strip()
print({'resolved_repo': str(REPO_ROOT), 'HEAD': COMMIT})

In [ ]:
from aic2026_eval.discovery import resolve_dataset_root, resolve_or_pack_archive

DATASET_ROOT = resolve_dataset_root(DATA_INPUT)
DRAFT_ZIP = resolve_or_pack_archive(DRAFT_INPUT, 'aic2026_dev_l21_150_draft_v1.zip', {'queries.jsonl','gt_provisional.jsonl','anchor_index_provisional.jsonl','manifest.json'}, '/kaggle/working/aic2026_dev_l21_150_draft_v1_input.zip')
DEV_CROSS_ZIP = resolve_or_pack_archive(CROSS_INPUT, 'aic2026_dev_cross_60_v1.zip', {'queries.jsonl','gt.jsonl','annotation_audit.jsonl','manifest.json'}, '/kaggle/working/aic2026_dev_cross_60_v1_input.zip', optional=True)
print({'resolved_dataset': str(DATASET_ROOT), 'resolved_l21_draft': str(DRAFT_ZIP), 'resolved_dev_cross': str(DEV_CROSS_ZIP) if DEV_CROSS_ZIP else None})

In [ ]:
from aic2026_eval.l21_finalize import run_l21_finalization

RESULT = run_l21_finalization(dataset_root=DATASET_ROOT, draft_zip=DRAFT_ZIP, dev_cross_zip=DEV_CROSS_ZIP, output_root=OUTPUT_ROOT, l21_zip_path=L21_ZIP, dev_zip_path=DEV_ZIP, git_commit=COMMIT)
print(json.dumps({key: value for key, value in RESULT.items() if key.isupper()}, ensure_ascii=False, indent=2))

In [ ]:
assert RESULT['L21_QUERY_COUNT'] == 150
assert RESULT['L21_KIS_COUNT'] == RESULT['L21_QA_COUNT'] == RESULT['L21_TRAKE_COUNT'] == 50
assert RESULT['L21_CANONICAL_ANCHORS'] == 99
assert RESULT['L21_FRAME_COORDINATE_CONTRACT'] in {'PASS','FAIL'}
if RESULT['DEV_L21_150_SCORING_READY'] == 'YES':
    assert L21_ZIP.is_file() and RESULT['L21_UNRESOLVED_ANCHORS'] == 0
    with ZipFile(L21_ZIP) as archive: assert {'queries.jsonl','gt.jsonl','manifest.json','annotation_audit.jsonl'} <= set(archive.namelist())
if RESULT['TEAM_DEV_BUNDLE'] == 'READY':
    assert DEV_ZIP.is_file()
    with ZipFile(DEV_ZIP) as archive:
        members = archive.namelist()
        assert not any('sealed' in name.lower() for name in members)
        assert not any(name.endswith(('.mp4','.npy','.npz','.pt','.pth','.bin')) for name in members)
print('Output validation: PASS')

In [ ]:
for key in ('TEAM_EVAL_L21_FINALIZATION','L21_QUERY_COUNT','L21_KIS_COUNT','L21_QA_COUNT','L21_TRAKE_COUNT','L21_FRAME_COORDINATE_CONTRACT','L21_CANONICAL_ANCHORS','L21_RESOLVED_ANCHORS','L21_UNRESOLVED_ANCHORS','DEV_L21_150_SCORING_READY','DEV_CROSS_60_STATUS','SEALED_FINAL_30_STATUS','TEAM_DEV_BUNDLE','RETURN_TO_MAIN_PIPELINE'):
    value = RESULT[key]
    if key.endswith('_COUNT') and value not in {150,50}: value = 'FAIL'
    print(f'{key}={value}')
print('L21 ZIP:', RESULT['l21_zip_path'])
print('TEAM DEV ZIP:', RESULT['dev_zip_path'])
print('REVIEW ZIP:', RESULT.get('review_zip_path'))
print('No new benchmark generation. No architecture work.')